# QMIX + PER + Reward Shaping — Smart Warehouse
Monotonic Value Function Factorisation for Multi-Agent RL.

**Run on Colab GPU: Runtime → Change runtime type → T4 GPU**

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
import os
if not os.path.exists('/content/smart-warehouse'):
    !git clone https://github.com/yjkim717/smart-warehouse.git
else:
    !cd /content/smart-warehouse && git pull origin main

!pip install rware gymnasium pyyaml -q

import sys
sys.path.insert(0, '/content/smart-warehouse')
os.chdir('/content/smart-warehouse')
print('Setup done.')

In [ ]:
# ── Run Training ──────────────────────────────────────────────────
# Uses train_qmix_per.py with PER + Reward Shaping (MAPPO-equivalent env)
!python scripts/train_qmix_per.py \
    --env-config configs/env_config.yaml \
    --qmix-config configs/qmix_config_per.yaml

In [ ]:
# ── Show Training Curve ───────────────────────────────────────────
from IPython.display import Image
Image('results/plots/qmix_per_training_curve.png')

In [ ]:
# ── Final Evaluation (300 episodes) ──────────────────────────────
import yaml, numpy as np, torch
import sys
sys.path.insert(0, '/content/smart-warehouse')

from src.env.warehouse_env import WarehouseEnv
from src.algorithms.qmix import QMIX
from src.analytics import RewardTracker

with open('configs/env_config.yaml') as f:
    env_config = yaml.safe_load(f)
with open('configs/qmix_config_per.yaml') as f:
    qmix_config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
env = WarehouseEnv(env_config)
qmix = QMIX(qmix_config, env.obs_dim, env.action_dim, env.n_agents, device=device)
qmix.load('results/checkpoints_per/qmix_per_best.pt')

tracker = RewardTracker(n_agents=env.n_agents)
MAX_STEPS = env_config['env'].get('max_steps', 500)

for ep in range(300):
    obs = env.reset()
    tracker.start_episode()
    for _ in range(MAX_STEPS):
        actions = qmix.select_actions(obs, explore=False)
        obs, rews, dones, _ = env.step(actions.tolist())
        tracker.record_step(rews)
        if all(dones): break
    tracker.end_episode()

env.close()
s = tracker.summary()
print('=' * 60)
print('  QMIX+PER Final Evaluation (300 episodes)')
print('=' * 60)
print(f"  Mean reward : {s['team_total_reward']['mean']:.4f}")
print(f"  Std         : {s['team_total_reward']['std']:.4f}")
print(f"  Min / Max   : {s['team_total_reward']['min']:.4f} / {s['team_total_reward']['max']:.4f}")
pos = sum(1 for e in tracker.episodes if e['team_total_reward'] > 0)
print(f"  Positive ep : {pos}/300 ({pos/3:.1f}%)")
print('=' * 60)
print(f"  Random baseline : 0.066")
print(f"  Greedy baseline : 0.443")
print(f"  MAPPO           : 8.69")
print(f"  QMIX+PER        : {s['team_total_reward']['mean']:.4f}")

In [ ]:
# ── Save & Download Results ───────────────────────────────────────
import json, os

os.makedirs('results/logs_per', exist_ok=True)
tracker.save('results/logs_per/qmix_per_eval_300ep.json')
tracker.save_csv('results/logs_per/qmix_per_eval_300ep.csv')

from google.colab import files
files.download('results/logs_per/qmix_per_eval_300ep.json')
files.download('results/logs_per/qmix_per_eval_curve.json')
files.download('results/plots/qmix_per_training_curve.png')
print('Download started.')